# Preprocessing and reproducible splits
Review the audit before creating splits. Corrupt images, unknown filename groups, and conflicting duplicate labels stop split creation.

In [ ]:
from pathlib import Path
import os
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "configs/cpu.yaml").exists():
    raise RuntimeError("Open this notebook from the project root or notebooks directory")
os.chdir(ROOT)
import yaml
from ml.runtime import configure
config = yaml.safe_load(Path("configs/cpu.yaml").read_text())
configure(config["seed"], config["threads"])

In [ ]:
import pandas as pd
from ml.data.dataset import build_manifest, XRayDataset
inventory = pd.read_csv("artifacts/data/inventory.csv").fillna("")
manifest = build_manifest(inventory, config["manifest"], config["seed"])
manifest.groupby(["split", "label"]).size().unstack(fill_value=0)

## Image transformations
Convert grayscale to RGB, bilinearly resize with aspect-ratio-preserving black padding to 224×224, scale to [0,1], then apply ImageNet channel means and standard deviations. Training adds rotations up to 5° and translation up to 2%. No flips, crops, or aggressive color changes.

In [ ]:
import matplotlib.pyplot as plt
import torch
from PIL import Image
from ml.data.transforms import preprocessing, MEAN, STD
row = manifest[manifest.split == "train"].iloc[0]
image = Image.open(Path(config["dataset_root"]) / row.path).convert("RGB")
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(image)
axes[0].set_title("Original")
for ax, training in zip(axes[1:], [False, True]):
    tensor = preprocessing(config["image_size"], training)(image)
    display = tensor * torch.tensor(STD)[:, None, None] + torch.tensor(MEAN)[:, None, None]
    ax.imshow(display.permute(1, 2, 0).clamp(0, 1))
    ax.set_title("Training augmentation" if training else "Evaluation input")
for ax in axes: ax.axis("off")
plt.show()

## Bounded-memory DataLoader
Batch size 8, no subprocesses, no pinned memory, and no in-memory dataset cache. Try batch size 4 or 2 if available RAM falls; benchmark workers 1–2 only if loading is a measured bottleneck.

In [ ]:
from ml.training.trainer import make_loader
dataset = XRayDataset(config["dataset_root"], manifest, "train", preprocessing(config["image_size"]))
images, labels = next(iter(make_loader(dataset, config)))
assert images.shape == (8, 3, config["image_size"], config["image_size"])
images.shape, labels.tolist()